**Daniel Yu & Jordan Wang**

Spring 2026

CS 443: Bio-inspired Machine Learning

# Extension 2: More sophisticated text preprocessing

This extension looks at a few simple preprocessing changes for the Skipgram text pipeline: stop-word removal, stemming, and lemmatization.

The goal is to keep the setup consistent with the rest of Project 3 by reusing the same `WordLevelDataset` and `text_util` helpers, then comparing how the different corpora change the vocabulary and training pairs.



In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from text_dataset_word import WordLevelDataset
from text_util import find_unique_word_counts, tokenize_words

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.show()
plt.rcParams.update({'font.size': 18})

np.set_printoptions(suppress=True, precision=7)

# Automatically reload external modules
%load_ext autoreload
%autoreload 2

2026-05-01 16:14:31.007213: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-01 16:14:32.085442: I tensorflow/core/platform/cpu_feature_guard.cc:211] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Extension 2: Preprocess IMDb reviews

This extension builds on the Skipgram pipeline by testing a few extra text preprocessing steps: removing stop words, stemming words, and lemmatizing words.

We kept the setup mostly the same as the rest of Project 3 by reusing the same WordLevelDataset and text\_util helper functions. This makes it easier to compare how each preprocessing method changes the vocabulary size and the number of training pairs.


In [5]:
imdb_ds = WordLevelDataset(verbose=False)
reviews_dev = imdb_ds.load(N_reviews=1000)
corpus_base = imdb_ds.make_corpus(reviews_dev, min_sent_size=2)

print(f'Number of reviews loaded: {len(reviews_dev)}')
print(f'Number of sentences in the baseline corpus: {len(corpus_base)}')
print(f'Baseline vocabulary size: {len(imdb_ds.make_vocabulary(corpus_base))}')
print('First baseline sentence:')
print(corpus_base[0][:20])

sample_sentence = tokenize_words(reviews_dev[0].split('.')[0])
print('First raw tokenized sentence:')
print(sample_sentence[:20])

Number of reviews loaded: 1000
Number of sentences in the baseline corpus: 11543
Baseline vocabulary size: 18572
First baseline sentence:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked']
First raw tokenized sentence:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked']


In [ ]:
        {
            "cell_type": "markdown",
            "id": "#VSC-25445c98",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "## Conclusion",
                "",
                "Preprocessing choices significantly change vocabulary size and training pairs. Choose preprocessing based on whether you prefer compact, generalized tokens or preserving detailed word distinctions."
            ]
        },
    }
    stemmer = None
    lemmatizer = None

corpus_nostop = [[word for word in sentence if word not in stop_words] for sentence in corpus_base]
corpus_stem = [[stemmer.stem(word) for word in sentence] for sentence in corpus_base] if stemmer is not None else corpus_base
corpus_lemma = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in corpus_base] if lemmatizer is not None else corpus_base
corpus_nostop_lemma = [[lemmatizer.lemmatize(word) for word in sentence if word not in stop_words] for sentence in corpus_base] if lemmatizer is not None else corpus_nostop

print(f'Baseline sentence example: {corpus_base[0][:15]}')
print(f'No stop words example: {corpus_nostop[0][:15]}')
print(f'Stemmed example: {corpus_stem[0][:15]}')
print(f'Lemmatized example: {corpus_lemma[0][:15]}')
print(f'No stop words + lemmatized example: {corpus_nostop_lemma[0][:15]}')

methods = {
    'baseline': corpus_base,
    'no_stop': corpus_nostop,
    'stem': corpus_stem,
    'lemma': corpus_lemma,
    'no_stop_lemma': corpus_nostop_lemma
}

rows = []
for method_name, corpus in methods.items():
    vocab = imdb_ds.make_vocabulary(corpus)
    word2ind = imdb_ds.make_word2ind_mapping(vocab)
    ind2word = imdb_ds.make_ind2word_mapping(vocab)
    target_words_int, context_words_int = imdb_ds.make_target_context_word_lists(corpus, word2ind, imdb_ds.context_win_sz)
    word_counts = find_unique_word_counts(corpus, sort_by_count=True)

    rows.append({
        'method': method_name,
        'sentences': len(corpus),
        'vocab_size': len(vocab),
        'pairs': len(target_words_int),
        'most_common_word': next(iter(word_counts)),
        'most_common_count': next(iter(word_counts.values()))
    })

summary_df = pd.DataFrame(rows)
summary_df

Baseline sentence example: ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be']
No stop words example: ['one', 'other', 'reviewers', 'has', 'mentioned', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'hooked']
Stemmed example: ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be']
Lemmatized example: ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be']
No stop words + lemmatized example: ['one', 'other', 'reviewers', 'has', 'mentioned', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'hooked']


,method,sentences,vocab_size,pairs,most_common_word,most_common_count
0,baseline,11543,18572,848358,the,13417
1,no_stop,11543,18532,519792,movie,1770
2,stem,11543,18572,848358,the,13417
3,lemma,11543,18572,848358,the,13417
4,no_stop_lemma,11543,18532,519792,movie,1770


In [7]:
# Compare preprocessing variants against baseline using the table built above (summary_df).
cmp_df = summary_df.copy()
baseline = cmp_df.loc[cmp_df['method'] == 'baseline'].iloc[0]

for col in ['vocab_size', 'pairs', 'sentences']:
    cmp_df[f'{col}_delta'] = cmp_df[col] - baseline[col]
    cmp_df[f'{col}_pct'] = 100.0 * cmp_df[f'{col}_delta'] / baseline[col]

cols_to_show = [
    'method',
    'sentences', 'sentences_delta', 'sentences_pct',
    'vocab_size', 'vocab_size_delta', 'vocab_size_pct',
    'pairs', 'pairs_delta', 'pairs_pct',
    'most_common_word', 'most_common_count'
 ]

cmp_df_display = cmp_df[cols_to_show].sort_values('method').reset_index(drop=True)
cmp_df_display[['sentences_pct', 'vocab_size_pct', 'pairs_pct']] = cmp_df_display[[
    'sentences_pct', 'vocab_size_pct', 'pairs_pct'
 ]].round(2)

print('Comparison relative to baseline (delta and % change):')
display(cmp_df_display)

# Quick text summary
smallest_vocab_row = cmp_df.loc[cmp_df['vocab_size'].idxmin()]
most_pairs_row = cmp_df.loc[cmp_df['pairs'].idxmax()]

print(f"- Smallest vocabulary: {smallest_vocab_row['method']} ({smallest_vocab_row['vocab_size']})")
print(f"- Most training pairs: {most_pairs_row['method']} ({most_pairs_row['pairs']})")

Comparison relative to baseline (delta and % change):


,method,sentences,sentences_delta,sentences_pct,vocab_size,vocab_size_delta,vocab_size_pct,pairs,pairs_delta,pairs_pct,most_common_word,most_common_count
0,baseline,11543,0,0.0,18572,0,0.00,848358,0,0.00,the,13417
1,lemma,11543,0,0.0,18572,0,0.00,848358,0,0.00,the,13417
2,no_stop,11543,0,0.0,18532,-40,-0.22,519792,-328566,-38.73,movie,1770
3,no_stop_lemma,11543,0,0.0,18532,-40,-0.22,519792,-328566,-38.73,movie,1770
4,stem,11543,0,0.0,18572,0,0.00,848358,0,0.00,the,13417


- Smallest vocabulary: no_stop (18532)
- Most training pairs: baseline (848358)


## Conclusion

We found that changing preprocessing (stop-word removal, stemming, lemmatization) changes vocabulary size slightly and the number of training pairs. Pick preprocessing based on whether you want a smaller, more generalized vocabulary or to preserve detailed word forms.